# 1. MetaQ: construct metacells from a small K562 sample

Run from the repository root with the **MetaQ-env** Python 3.10 kernel after the MetaQ installation and patch in the repository README. This notebook is for metacell construction; it is not required to use the pretrained model in notebooks 02–04.

Download only the required K562 input (about 2 GiB; download time is additional):

If access is restricted, run `hf auth login` once before downloading.

```bash
python -m pip install huggingface_hub
hf download netbiolab/MetacellFormer-evaluation inputs/perturbation/K562.h5ad --repo-type dataset --local-dir ../MetacellFormer-evaluation
```

The default input directory is `../MetacellFormer-evaluation`; set `METACELLFORMER_EVALUATION` if yours differs. The input is single-cell unnormalized UMI counts. The notebook uses the first 100 cells and runs MetaQ for 300 epochs on **CPU** (about 1 minute on the tested two-thread CPU).

Expected files under `example_outputs/metaq/`: `K562_100_cells.h5ad` (100 × 8,563), `run/memberships.tsv` (100 rows), and `metacells.h5ad` (2 × 8,563). Completion means every source cell has a membership and each metacell's raw counts exactly equal the sum of its member cells. The numbers above describe this small example, not a paper benchmark score.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

repo = Path.cwd().resolve()
assert (repo / "scripts/prepare_metaq.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
source = data_root / "inputs/perturbation/K562.h5ad"
assert source.is_file(), f"Download the evaluation dataset: {source}"
work = repo / "example_outputs/metaq"
work.mkdir(parents=True, exist_ok=True)
raw_file = work / "K562_100_cells.h5ad"

backed = ad.read_h5ad(source, backed="r")
try:
    raw = backed[:100].to_memory()
finally:
    backed.file.close()
counts = sparse.csr_matrix(raw.X)
assert raw.n_obs == 100 and raw.obs_names.is_unique
assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
assert np.allclose(counts.data, np.rint(counts.data))
raw.X = counts
raw.write_h5ad(raw_file)
print(raw.shape)

(100, 8563)


In [2]:
env = os.environ.copy()
env.update(METAQ_NUM_WORKERS="0", METAQ_PIN_MEMORY="0", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
completed = subprocess.run([sys.executable, str(repo / "scripts/prepare_metaq.py"),
                "--input", str(raw_file), "--output-dir", str(work / "run"),
                "--device", "cpu", "--batch-size", "16"],
               cwd=repo, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    raise RuntimeError(completed.stdout[-4000:])
completed = subprocess.run([sys.executable, str(repo / "scripts/aggregate_metacells.py"),
                str(raw_file), str(work / "run/memberships.tsv"),
                str(work / "metacells.h5ad"), "--source-id", "K562_example"],
               cwd=repo, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    raise RuntimeError(completed.stdout[-4000:])

In [3]:
membership = pd.read_csv(work / "run/memberships.tsv", sep="\t", dtype=str)
metacells = ad.read_h5ad(work / "metacells.h5ad")
assert membership.source_cell_id.tolist() == raw.obs_names.tolist()
assert int(metacells.obs.n_cells.sum()) == raw.n_obs
assert metacells.var_names.equals(raw.var_names)
for group in metacells.obs_names:
    selected = membership.metacell_id.to_numpy() == group
    expected = np.asarray(counts[selected].sum(axis=0)).ravel()
    np.testing.assert_array_equal(metacells[group].X.toarray().ravel(), expected)
print("Verified", len(metacells), "metacells from", len(raw), "source cells")

Verified 2 metacells from 100 source cells
